In [56]:
from dlfs.base import Module, Loss, Optimizer, Layer

from dlfs.modules import SequentialWrapper
from dlfs.layers import DenseLayer, ConvolutionalLayer
from dlfs.activation import ReLU, Sigmoid

from dlfs.loss import MSE_Loss, BCE_Loss
from dlfs.optimizers import Optimizer_Adam
from dlfs.helpers import dilate, pad_to_shape

import numpy as np
import matplotlib.pyplot as plt
from scipy import signal

In [ ]:
def im2col_multi(X, kernel_shape, stride=1, padding=(0, 0)):
    B = X.shape[0]
    kH, kW = kernel_shape

    if isinstance(padding, tuple):
        pad_H, pad_W = padding
    else:
        pad_H = pad_W = padding

    X_padded = np.pad(X, ( (0, 0), (0, 0),(pad_H, pad_H), (pad_W, pad_W) ), mode='constant')

    H_p, W_p = X_padded.shape[2:]

    out_H = (H_p- kH) // stride + 1
    out_W = (W_p- kW) // stride + 1

    cols = []

    for b in range(B):
        for i in range(0, out_H*stride, stride):
            for j in range(0, out_W * stride, stride):
                patch = X_padded[b, :, i:i+kH, j:j+kW].ravel()
                cols.append(patch)
    
    return np.array(cols), out_H, out_W

def col2im_multi(cols, output_shape, kernel_shape, stride=1, padding=0):
    B, C, H, W = output_shape
    kH, kW = kernel_shape
    H_p, W_p = H+2*padding, W+2*padding
    X_padded = np.zeros((B, C, H_p, W_p))

    out_H = (H_p - kH)//stride + 1
    out_W = (W_p - kW)//stride + 1

    idx = 0
    for b in range(B):
        for i in range(0, out_H*stride, stride):
            for j in range(0, out_W*stride, stride):
                patch = cols[idx].reshape(C, kH, kW)
                X_padded[b, :, i:i+kH, j:j+kW] += patch
                idx += 1

    if padding>0:
        X_padded = X_padded[:, :, padding:-padding, padding:-padding]

    return X_padded

def conv2d(X, W, stride=1, padding=0):
    
    C_out, C_in, kH, kW = W.shape
    B = X.shape[0]
    X_col, out_H, out_W = im2col_multi(X, (kH, kW), stride, padding)

    W_col = W.reshape(C_out, -1)
    Y_col = X_col @ W_col.T

    Y = Y_col.T.reshape(B, C_out, out_H, out_W)
    return Y

def conv_transpose2d(Y, W, stride=1, padding=0, output_shape=None):
    C_out, C_in, kH, kW = W.shape
    B = Y.shape[0]
    Y_col = Y.reshape(C_out, -1)
    W_col = W.reshape(C_out, -1)
    X_col = W_col.T @ Y_col

    if output_shape is None:
        H_out = (Y.shape[2]-1) * stride - 2*padding + kH
        W_out = (Y.shape[3]-1) * stride - 2*padding + kW
        output_shape = (B, C_in, H_out, W_out)

    X = col2im_multi(X_col.T, output_shape=output_shape, kernel_shape=(kH, kW), stride=stride, padding=padding)

    return X

In [ ]:
class ConvTransposeLayer(Layer):

    def __init__(self, input_channels: tuple, output_channels: int, kernel_size: int, stride: int = 1, padding: int = 0, output_padding=0) -> None:

        self.input_channels = input_channels
        self.output_channels = output_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        self.output_padding = output_padding

        # Create output and kernel shapes
        self.kernel_size = kernel_size

        # Initialize layer parameters
        self.kernels = np.random.randn(output_channels, input_channels, kernel_size, kernel_size)
        self.biases = np.random.randn(output_channels)

    def forward(self, inputs: np.ndarray, training=False) -> None:
        self.output = conv_transpose2d(inputs, self.kernels, stride=self.stride, padding=self.padding)


    def backward(self, delta: np.ndarray) -> None:
        pass

    def get_parameters(self):
        param_names = ["kernels", "biases"]
        return super()._filter_parameters(param_names)

    def _calculate_kernel_gradient(self, inputs: np.ndarray, delta: np.ndarray, kernel: np.ndarray, stride: int = 1) -> np.ndarray:
        """
        Helper method for calculating kernel gradient.

        Parameters
        ----------
        inputs : np.ndarray
            Current sample the gradient is calculated for.

        delta : np.ndarray
            Accumulated gradient obtained by backpropagation.

        kernel : np.ndarray
            Kernel used in convolutional layer.

        stride : int, default=1
            Step size at which the kernel moves across the input.

        Returns
        -------
        kernel_grad : np.ndarray
            Kernel gradient.
        """

        if stride > 1:

            # If stride is present delta needs to be dilated
            delta_dilated = dilate(delta, stride)

            delta_dilated_height, delta_dilated_width = delta_dilated.shape[-2:]
            input_height, input_width = inputs.shape[-2:]
            kernel_shape = kernel.shape[-1]

            if delta_dilated_height == input_height - kernel_shape + 1 and delta_dilated_width == input_width - kernel_shape + 1:
                # If dilated delta shape matches the needed correlation shape gradient can be computed
                dkernel = signal.correlate2d(inputs, delta_dilated, "valid")
            else:
                # If dilated delta shape doesn't match the needed correlation shape padding is needed
                new_delta_shape = (input_height - kernel_shape + 1, input_width - kernel_shape + 1)
                delta_dilated_padded = pad_to_shape(delta_dilated, new_delta_shape)
                dkernel = signal.correlate2d(inputs, delta_dilated_padded, "valid")

        else:
            # Gradient with respect to kernel is valid cross correlation between inputs and delta
            dkernel = signal.correlate2d(inputs, delta, "valid")

        return dkernel

    def _calculate_input_gradient(self, inputs: np.ndarray, delta: np.ndarray, kernel: np.ndarray, stride: int = 1):
        """
        Helper method for calculating input gradient.

        Parameters
        ----------
        inputs : np.ndarray
            Current sample the gradient is calculated for.

        delta : np.ndarray
            Accumulated gradient obtained by backpropagation.

        kernel : np.ndarray
            Kernel used in convolutional layer.

        stride : int, default=1
            Step size at which the kernel moves across the input.

        Returns
        -------
        input_grad : np.ndarray
            Input gradient.
        """

        if stride > 1:

            delta_dilated = dilate(delta, stride)

            delta_dilated_height, delta_dilated_width = delta_dilated.shape[-2:]
            input_height, input_width = inputs.shape[-2:]
            kernel_shape = kernel.shape[-1]

            if delta_dilated_height == input_height - kernel_shape + 1 and delta_dilated_width == input_width - kernel_shape + 1:
                # If dilated delta shape matches the needed coonvolution shape gradient can be computed
                dinput = signal.convolve2d(delta_dilated, kernel, "full")
            else:
                # If dilated delta shape doesn't match the needed convolution shape padding is needed
                new_delta_shape = (input_height - kernel_shape + 1, input_width - kernel_shape + 1)
                delta_dilated_padded = pad_to_shape(delta_dilated, new_delta_shape)
                dinput = signal.convolve2d(delta_dilated_padded, kernel, "full")

        else:
            # Gradient with respect to inputs is full convolution between delta and kernel
            dinput = signal.convolve2d(delta, kernel, "full")

        return dinput

    def _calculate_kernel_gradient_transpose(self, inputs: np.ndarray, delta: np.ndarray, kernel: np.ndarray,
                                            stride: int = 1, padding: int = 0) -> np.ndarray:
        """
        Helper for ConvTranspose2d: gradient wrt kernel.

        Parameters
        ----------
        inputs : np.ndarray
            Input feature map (before transposed convolution).
        delta : np.ndarray
            Gradient of loss wrt layer output (grad_output).
        kernel : np.ndarray
            Kernel used in this layer.
        stride : int
            Stride used in forward.
        padding : int
            Padding used in forward.

        Returns
        -------
        dkernel : np.ndarray
            Gradient wrt kernel weights.
        """

        if stride > 1:

            # If stride is present delta needs to be dilated
            inputs_dilated = dilate(inputs, stride)

            inputs_dilated_height, inputs_dilated_width = inputs_dilated.shape[-2:]
            delta_height, delta_width = delta.shape[-2:]
            kernel_shape = kernel.shape[-1]

            if inputs_dilated_height == delta_height - kernel_shape + 1 and inputs_dilated_width == delta_width - kernel_shape + 1:
                # If dilated delta shape matches the needed correlation shape gradient can be computed
                dkernel = signal.correlate2d(inputs_dilated, delta, "valid")
            else:
                # If dilated delta shape doesn't match the needed correlation shape padding is needed
                new_inputs_shape = (delta_height - kernel_shape + 1, delta_width - kernel_shape + 1)
                inputs_dilated_padded = pad_to_shape(inputs_dilated, new_inputs_shape)
                dkernel = signal.correlate2d(inputs_dilated_padded, delta, "valid")

        else:
            # Gradient with respect to kernel is valid cross correlation between inputs and delta
            dkernel = signal.correlate2d(inputs, delta, "valid")

        """        
        # 1. Dilate the input
        input_dilated = dilate(inputs, stride)

        # 2. Pad the dilated input (as in forward)
        if padding > 0:
            input_dilated = np.pad(input_dilated, pad_width=padding)

        # 3. Cross-correlation of input_dilated and grad_output
        dkernel = signal.correlate2d(input_dilated, delta, mode='valid')
        """

        return dkernel

    def _calculate_input_gradient_transpose(self, inputs: np.ndarray, delta: np.ndarray, kernel: np.ndarray,
                                        stride: int = 1, padding: int = 0) -> np.ndarray:
        """
        Helper for ConvTranspose2d: gradient wrt inputs.

        Parameters
        ----------
        inputs : np.ndarray
            Input feature map (before transposed convolution).
        delta : np.ndarray
            Gradient of loss wrt layer output (grad_output).
        kernel : np.ndarray
            Kernel used in this layer.
        stride : int
            Stride used in forward.
        padding : int
            Padding used in forward.

        Returns
        -------
        dinput : np.ndarray
            Gradient wrt layer input (to pass backward).
        """

        # 1. Flip kernel 180 degrees
        kernel_rot = np.rot90(kernel, 2)

        # 2. Full convolution of grad_output with flipped kernel
        dinput = signal.convolve2d(delta, kernel_rot, mode='valid')

        # 3. Remove padding that was added in forward
        if padding > 0:
            dinput = dinput[padding:-padding, padding:-padding]

        # 4. Undilate (reverse the stride)
        if stride > 1:
            dinput = dinput[::stride, ::stride]

        return dinput

In [61]:
class ConvTranspose2D:
    def __init__(self, input_channels, output_channels, kernel_size, stride=1, padding=0):
        self.input_channels = input_channels
        self.output_channels = output_channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.padding = padding
        
        # Randomly initialize kernels and biases
        self.kernels = np.random.randn(output_channels, input_channels, kernel_size, kernel_size)
        self.biases = np.random.randn(output_channels)

    def forward(self, inputs):
        """
        Forward pass for transposed convolution.

        Parameters
        ----------
        inputs : np.ndarray
            Input of shape (batch_size, input_channels, H, W)

        Returns
        -------
        np.ndarray
            Output of shape (batch_size, output_channels, H_out, W_out)
        """
        n_samples, _, H, W = inputs.shape
        
        # Compute output shape
        H_out = (H - 1) * self.stride - 2 * self.padding + self.kernel_size
        W_out = (W - 1) * self.stride - 2 * self.padding + self.kernel_size
        
        # Initialize output
        output = np.zeros((n_samples, self.output_channels, H_out, W_out))
        
        # Loop over batch, output channels, input channels
        for i in range(n_samples):
            for j in range(self.output_channels):
                for k in range(self.input_channels):
                    # Upsample input by stride
                    in_up = self.upsample_by_stride(inputs[i, k])
                    
                    # Add padding if specified
                    if self.padding > 0:
                        in_up = np.pad(in_up, self.padding, mode='constant')
                    
                    # Convolve using full mode (spreads input over output)
                    # Either rotate kernel and use correlate2d OR use convolve2d directly
                    output[i, j] += signal.convolve2d(in_up, self.kernels[j, k], mode='valid')
        
        return output

In [62]:
x = np.random.randn(5, 3, 27, 36)

layer1 = ConvolutionalLayer(input_shape=(3, 27, 36), output_channels=6, kernel_size=3, stride=2, padding=1)
layer2 = ConvTransposeLayer(input_channels = 6, output_channels = 3, kernel_size=3, stride=2, padding=3, output_padding=(4, 5))

layer1.forward(x)

print(layer1.output.shape)

layer2.forward(layer1.output)

print(layer2.output.shape)

(5, 6, 14, 18)
(5, 3, 27, 36)


In [63]:
delta = np.random.randn(*layer2.output.shape)

layer2.backward(delta)

ValueError: Target shape must be larger than the array shape.